---
## 🔧 Module 1: Installing LangChain (5 min)

### What We'll Install:
- `langchain` - Core LangChain library
- `langchain-openai` - OpenAI integration
- `python-dotenv` - Environment variable management

### Step 1.1: Install Required Packages

In [ ]:
# Install all required packages
%pip install langchain langchain-openai python-dotenv

### Step 1.2: Verify Installation

In [5]:
# Verify installations

In [6]:
pip show langchain

Name: langchain
Version: 1.0.8
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: d:\mentoring\learwithsarvesh\.venv\lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: langchain-tools
Note: you may need to restart the kernel to use updated packages.


In [1]:
import langchain
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

print(f"LangChain version: {langchain.__version__}")
print("✅ All packages imported successfully!")

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LangChain version: 1.0.8
✅ All packages imported successfully!


### Step 1.3: Setup Environment Variables

**Creating .env file in VS Code:**
1. Create a new file called `.env` in your project root
2. Add your OpenAI API key:
   ```
   OPENAI_API_KEY=sk-your-actual-key-here
   ```
3. Save the file

**Important:** Never commit `.env` to GitHub! Add it to `.gitignore`

In [2]:
# Load environment variables from .env file

import os
from dotenv import load_dotenv, find_dotenv

# Load the .env file
load_dotenv(find_dotenv(), override=True)

# Verify the API key is loaded (show first 10 chars only for security)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"✅ API Key loaded: {api_key[:10]}...")
else:
    print("❌ API Key not found! Check your .env file")

✅ API Key loaded: sk-proj-3n...


---
## 📝 Module 2: PromptTemplate Basics (20 min)

### What is a Prompt Template?
A **Prompt Template** is a reusable blueprint for creating prompts with variables.

**Benefits:**
- Reusability across different inputs
- Consistency in prompt structure
- Easy maintenance and updates
- Professional prompt engineering

### Step 2.1: Simple Single-Variable Template

In [3]:
# Import PromptTemplate and create a simple template

from langchain_core.prompts import PromptTemplate


# Create a simple template with one variable
simple_prompt = PromptTemplate.from_template(
    "Explain {topic} in simple words."
)

# Format the template
formatted = simple_prompt.format(topic="quantum physics")
print(formatted)

Explain quantum physics in simple words.


### Step 2.2: Multi-Variable Templates

Real applications need multiple variables in prompts.

In [4]:
# Multi-variable template

email_prompt = PromptTemplate.from_template(
    "Write a {tone} email to {person} about {subject}"
)


# Format with multiple variables:
email = email_prompt.format(
    tone="formal",
    person="the HR team",
    subject="updating my bank details"
)

print(email)

Write a formal email to the HR team about updating my bank details


### Step 2.3: Passing Prompts to LLM

Now let's connect our template to an actual LLM and get responses!

In [5]:
# Initialize the LLM

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = "gpt-4o-mini", temperature = 0.7)

# Format prompt and get response:
topic_prompt = simple_prompt.format(topic="machine learning")
response = llm.invoke(topic_prompt)

print("Response:")
print(response.content)

Response:
Machine learning is a branch of artificial intelligence that allows computers to learn from data and make decisions or predictions without being explicitly programmed for every task. Imagine teaching a child to recognize animals: instead of giving them a detailed description of each animal, you show them many pictures of cats and dogs. Over time, the child learns to identify each animal based on what they’ve seen.

In the same way, machine learning uses large amounts of data to help computers recognize patterns and make guesses about new data. For example, it can help an email program recognize spam by learning from examples of what spam looks like and what doesn’t. The more data it processes, the better it gets at making accurate predictions or decisions.


### Step 2.4: Reusable Template Examples

Let's create practical, reusable templates for common tasks.

In [6]:
# Create email writer, summarizer, and code explainer templates
email_writer = PromptTemplate.from_template(
    """Write a {tone} email about {topic}. 
    
    The email should be addressed to {recipient} and should be approximately {length} sentences long."""
)

# Example2: Text Summarizer Template:
summarizer = PromptTemplate.from_template(
    "Summarize the following text in {num_sentences} sentences:\n\n{text}"
)

# Example 3: Code Explainer Template
code_explainer = PromptTemplate.from_template(
    "Explain this {language} code to a {level} developer:\n\n{code}"
)

#Test one template:
email_output = email_writer.format(
    tone = "sarcastic",
    topic="project deadline extension",
    recipient = "Project Manager",
    length = "3"
)


response = llm.invoke(email_output)
print("Generated Email:")
print(response.content)

Generated Email:
Subject: Oh Joy, Another Extension!

Dear [Project Manager's Name],

I must say, receiving yet another deadline extension for our project is just the thrilling twist I never knew I needed in my life. Who doesn't love the excitement of procrastination and last-minute panic? I can hardly contain my enthusiasm for this newfound opportunity to stretch my stress levels even further!

Best,  
[Your Name]


### Step 2.5: Understanding .format() Method

The `.format()` method is crucial - it fills in the template variables.

In [7]:
# Demonstrating .format() in detail

template = PromptTemplate.from_template(
    "You are a {role}. {task}"
)

# Way1: Using named arguments:
output1 = template.format(role="teacher", task="Explain Python to beginners")
print("Output 1:", output1)

# Way 2: Using a dictionary
inputs = {"role": "data scientist", "task": "Analyze this dataset"}
output2 = template.format(**inputs)
print("\nOutput 2:", output2)

# Way 3: Dynamic formatting
roles = ["chef", "doctor", "engineer"]

for role in roles:
    output = template.format(role=role, task="Describe your typical day")
    print(f"\n{role.upper()}: {output}")

Output 1: You are a teacher. Explain Python to beginners

Output 2: You are a data scientist. Analyze this dataset

CHEF: You are a chef. Describe your typical day

DOCTOR: You are a doctor. Describe your typical day

ENGINEER: You are a engineer. Describe your typical day


---
## 🎓 Module 3: Advanced Prompt Techniques (20 min)

### What You'll Learn:
- **Few-Shot Prompting** - Teaching by example
- **Role Prompting** - Setting AI persona
- **Instruction Prompting** - Detailed style control
- **Output Formatting** - Structured responses

### Step 3.1: Few-Shot Prompting

Few-shot learning provides examples to guide the AI's responses.

In [9]:
# Few-shot prompt for sentiment analysis
 
few_shot_prompts = PromptTemplate.from_template("""
Classify the sentiment of the text as Positive, Negative, or Neutral.

Examples:
- "I absolutely love this product!" → Positive
- "This is terrible and disappointing." → Negative
- "It's okay, nothing special." → Neutral

Now classify:
"{text}" →               
""")

# Test with different examples

test_texts = [
    "This is the best purchase I've ever made!",
    "Waste of money, very poor quality.",
    "It works fine, does what it's supposed to do."
]


for text in test_texts:
    formatted = few_shot_prompts.format(text = text)
    result = llm.invoke(formatted)
    print(f"Text: {text}")
    print(f"Sentiment: {result.content}\n")

Text: This is the best purchase I've ever made!
Sentiment: Positive

Text: Waste of money, very poor quality.
Sentiment: Negative

Text: It works fine, does what it's supposed to do.
Sentiment: Neutral



### Step 3.2: Role Prompting

Assigning a specific role/persona to the AI for consistent behavior.

In [ ]:
# Role-based prompting
role_prompt = PromptTemplate.from_template("""
You are a {role} with {years} of experience.

Your task: {task}

Provide a detailed, professional response from your perspective.                             
"""
)

# Example: Senior Developer
dev_response = llm.invoke(role_prompt.format(
    role="Senior Software Developer",
    years = "10",
    task = "Exaplin why code reviews are important"
))

print("== Senior Developer Repose ==")
print(dev_response.content)




== Senior Developer Repose ==
As a Senior Software Developer with a decade of experience, I have come to appreciate the critical role that code reviews play in the software development process. Code reviews are not merely an exercise in quality control; they are an integral part of a robust development workflow that fosters collaboration, knowledge sharing, and continuous improvement. Here are several reasons why code reviews are important:

### 1. **Improved Code Quality**
One of the primary benefits of code reviews is the enhancement of code quality. Reviewing code allows developers to identify bugs, inconsistencies, and potential performance issues before they make it to production. Multiple sets of eyes on code can catch mistakes that might be overlooked by the original author, leading to more reliable and maintainable software.

### 2. **Knowledge Sharing and Team Collaboration**
Code reviews serve as an excellent opportunity for team members to share knowledge. When developers re

In [11]:
# Example 2: Marketing Expert
marketing_response = llm.invoke(role_prompt.format(
    role="Marketing Expert",
    years="8",
    task="Create a catchy tagline for an AI-powered app"
))

print("\n=== Marketing Expert Response ===")
print(marketing_response.content)


=== Marketing Expert Response ===
As a marketing expert with eight years of experience, crafting a catchy tagline for an AI-powered app requires a deep understanding of the app's purpose, target audience, and unique value proposition. A tagline should be memorable, concise, and reflective of the app's capabilities while resonating with potential users.

### Considerations for Tagline Creation:

1. **Identify the Core Functionality**: Understand what the app does. Is it focused on productivity, fitness, education, entertainment, or something else? 

2. **Highlight the AI Aspect**: Emphasize how AI enhances the user experience, whether through personalization, automation, efficiency, or insights.

3. **Know Your Audience**: Tailor the language and tone to appeal to the intended user base, whether they are tech-savvy millennials, busy professionals, or families.

4. **Create an Emotional Connection**: People tend to remember phrases that evoke emotions or a sense of aspiration.

### Samp

### Step 3.3: Instruction Prompting with Style Control

Detailed instructions for controlling output format, tone, and length.

In [ ]:
# Instruction prompt with style control



### Step 3.4: Output Formatting

Controlling structured output (JSON, lists, tables, etc.)

In [ ]:
# Output formatting prompt



---
## 💬 Module 4: Chat Models - invoke, batch, stream (25 min)

### Three Ways to Interact with LLMs:
1. **invoke()** - Single synchronous call
2. **batch()** - Multiple calls in parallel
3. **stream()** - Token-by-token streaming

### Step 4.1: Using .invoke() - Single Request

In [ ]:
# .invoke() - Single request



### Step 4.2: Using .batch() - Multiple Parallel Requests

When you need to process multiple inputs efficiently.

In [ ]:
# .batch() - Multiple requests in parallel



### Step 4.3: Using .stream() - Token Streaming

Get responses token-by-token for real-time user experience (like ChatGPT interface).

In [ ]:
# .stream() - Token-by-token streaming



---
## 🔗 Module 5: LLMChain Deep Dive (25 min)

### What is an LLMChain?
An **LLMChain** combines a PromptTemplate + LLM into a single reusable component.

**Benefits:**
- Cleaner code structure
- Easy to reuse and share
- Better error handling
- Access to chain result objects

### Step 5.1: Basic LLMChain with Single Variable

In [ ]:
# Create an LLMChain



### Step 5.2: LLMChain with Multiple Variables

In [ ]:
# Multi-variable LLMChain



### Step 5.3: Understanding Chain Result Objects

Chains return rich result objects with metadata.

In [ ]:
# Get full chain result (not just output)



### Step 5.4: Input/Output Keys

Understanding and customizing chain input/output keys.

In [ ]:
# Checking input/output keys



### Step 5.5: Error Handling in Chains

Proper error handling for production applications.

In [ ]:
# Error handling example



---
## 🔀 Module 6: Sequential Chains (25 min)

### What are Sequential Chains?
Chains that run multiple steps in sequence, where the output of one chain becomes the input to the next.

**Two Types:**
1. **SimpleSequentialChain** - Single input/output per step
2. **SequentialChain** - Multiple inputs/outputs per step

### Step 6.1: SimpleSequentialChain

Perfect for simple workflows where each step has one input and one output.

In [ ]:
# SimpleSequentialChain example



### Step 6.2: SequentialChain with Multiple Inputs/Outputs

More powerful - handles multiple variables at each step.

In [ ]:
# SequentialChain with multiple inputs/outputs



### Step 6.3: Understanding Intermediate Outputs

How data flows between chains in a sequence.

In [ ]:
# Visualizing data flow in sequential chains



---
## 🚀 Module 7: Mini Project - Translate + Summarize Chain (20 min)

### Project Goal:
Build a practical application that:
1. Takes text in any language
2. Translates it to English
3. Summarizes the translated text
4. Returns both translation and summary

### Step 7.1: Build the Translation Chain

In [ ]:
# Step 1: Build the Translation Chain



### Step 7.2: Build the Summarization Chain

In [ ]:
# Step 2: Build the Summarization Chain



### Step 7.3: Combine into Complete Sequential Chain

In [ ]:
# Step 3: Combine both chains



### Step 7.4: Create a Reusable Function

Package the chain into a clean, reusable function for production use.

In [ ]:
# Production-ready function



---
## 📚 Course Recap & Summary

### 🎯 What We Covered Today (2.5 Hours)

#### ✅ Module 1: Installing LangChain (5 min)
- Installed `langchain`, `langchain-openai`, `python-dotenv`
- Set up environment variables and API keys
- Verified installation

#### ✅ Module 2: PromptTemplate Basics (20 min)
- Created single and multi-variable templates
- Passed prompts to LLMs
- Built reusable templates (email writer, summarizer, code explainer)
- Mastered the `.format()` method

#### ✅ Module 3: Advanced Prompt Techniques (20 min)
- **Few-shot prompting** - Teaching by example
- **Role prompting** - Setting AI personas
- **Instruction prompting** - Detailed style control
- **Output formatting** - Structured responses (JSON, lists)

#### ✅ Module 4: Chat Models (25 min)
- `.invoke()` - Single synchronous requests
- `.batch()` - Parallel processing for efficiency
- `.stream()` - Token-by-token streaming

#### ✅ Module 5: LLMChain Deep Dive (25 min)
- Created LLMChains with single and multiple variables
- Understood chain result objects
- Worked with input/output keys
- Implemented proper error handling

#### ✅ Module 6: Sequential Chains (25 min)
- **SimpleSequentialChain** - Single input/output per step
- **SequentialChain** - Multiple inputs/outputs per step
- Built complex multi-step workflows
- Managed intermediate outputs

#### ✅ Module 7: Mini Project (20 min)
- Built a complete Translate + Summarize application
- Combined multiple chains
- Created production-ready functions
- Tested with multiple languages

---

### 🚀 Key Takeaways

1. **Prompt Templates** make your prompts reusable and maintainable
2. **Advanced techniques** (few-shot, role, instruction) improve AI responses
3. **Chat models** offer flexibility with invoke, batch, and stream
4. **LLMChains** combine prompts and LLMs into clean components
5. **Sequential Chains** enable complex multi-step workflows
6. **Real projects** combine all concepts into practical applications

---

### 💡 Next Steps

- Build your own chain-based applications
- Explore LangChain Agents (next level!)
- Experiment with different LLM providers
- Try memory and conversation chains
- Connect to external APIs and databases

---

### 📖 Additional Resources

- **LangChain Documentation**: https://python.langchain.com/
- **OpenAI API Docs**: https://platform.openai.com/docs
- **LangChain Community**: https://github.com/langchain-ai/langchain

---

## 🎉 Congratulations!

You've completed the LangChain Complete Course! You now have the skills to:
- Build production-ready AI applications
- Create complex chains and workflows
- Handle multiple LLM interactions efficiently
- Structure code professionally

**Keep building and experimenting! 🚀**